In [1]:
import pandas as pd
from pathlib import Path
import numpy as np

import matplotlib.pyplot as plt
from matplotlib.pyplot import figure
import ase
import os
import ase.io

In [2]:
def get_freq_phonon_2x2x2_interpolated_DOS(file_name,num_atoms):
    cm_to_eV = 0.00012398425731484318
    
    with open(file_name,"r") as fp:
        lines = fp.readlines()
        
    freq = []
    tot_phdos = []
    site_proj_phdos = [[] for i in range(num_atoms)]
    for i in range(1,len(lines)):
        line = lines[i].split()
        freq.append(float(line[0])*cm_to_eV*1000)
        tot_phdos.append(float(line[1])/(cm_to_eV*1000))
        for j in range(0,num_atoms):
            site_proj_phdos[j].append(float(line[2+j])/(cm_to_eV*1000))
        
    return freq, tot_phdos, site_proj_phdos
def get_size(df):
    return len(df.structure)


In [3]:
# freq, tot_phdos, site_proj_phdos = get_freq_phonon_2x2x2_interpolated_DOS('/blue/hennig/jasongibson/elemental_sub/materials/Be/mp_relaxed/1608/phonon_2x2x2/inter/.dos',10)
# freq2, tot_phdos2, site_proj_phdos2 = get_freq_phonon_2x2x2_interpolated_DOS('/blue/hennig/jasongibson/elemental_sub/materials/Be/mp_relaxed/1608/phonon_2x2x2/.dos',10)

In [4]:
par_el = 'run_full'
root = '/blue/hennig/jasongibson/diff_model'
root = f'{root}/materials/{par_el}/mp_relaxed/'
df = pd.read_pickle(f'pkl_files/df_{par_el}_pred_m3gnet_eah.pkl')

In [5]:
df.shape

(6173, 28)

In [6]:
from tqdm.notebook import tqdm
tqdm.pandas()

In [7]:
# root = f'/blue/hennig/jasongibson/elemental_sub/materials/{par_el}/mp_relaxed/'

def get_phonon_properties(df):
    jobdir = f'{root}{df.name}/'
    filename = jobdir + "/relax.out"
    
    try:
        atom_obj = ase.io.read(filename)
    except:
        return pd.Series([np.nan, np.nan, np.nan], index=['freq', 'phdos', 'site_phdos'])
    
    if os.path.isfile(jobdir + 'phonon_2x2x2/.dos'):
        filename = jobdir + "/relax.out"
        atom_obj = ase.io.read(filename)
        freq, phdos, site_proj_phdos = get_freq_phonon_2x2x2_interpolated_DOS(
            jobdir + "/phonon_2x2x2/" + ".dos", atom_obj.get_global_number_of_atoms()
        )
    else:
        freq, phdos, site_proj_phdos = np.nan, np.nan, np.nan
    
    return pd.Series([freq, phdos, site_proj_phdos], index=['freq', 'phdos', 'site_phdos'])

df[['freq', 'phdos', 'site_phdos']] = df.progress_apply(get_phonon_properties, axis=1)


  0%|          | 0/6173 [00:00<?, ?it/s]

In [8]:
print(df.shape)
df.dropna(axis=0,inplace=True)
df.shape

(6173, 31)


(2486, 31)

In [9]:
import math

In [10]:
def get_dyn_stable(df):
    for freq, phdos in zip(df.freq,df.phdos):
        if (freq < 0) & (phdos > 0.0):
            return False
        if math.isnan(phdos):
            return False
    return True
df['dyn_stable'] = df.apply(get_dyn_stable,axis=1)

In [11]:
# plt.plot(df.loc[159493].freq,df.loc[159493].phdos)

In [12]:
df = df.loc[df.dyn_stable]
df.shape

(1818, 32)

In [13]:
def get_size(df):
    return len(df.structure)
df['natoms'] = df.apply(get_size, axis = 1)
df.loc[df.natoms<=12].shape

(1818, 32)

In [14]:
df.to_pickle(f'pkl_files/df_dyn_stab_{par_el}_strict.pkl')